# Stickman Studio · Voice server (Kaggle)

The same free voice server as the Colab notebook, on **Kaggle's free GPU** (about 30 GPU hours a week). Use it when Colab's free GPU is used up. It works from a phone browser too.

**First time only**
- Sign in to kaggle.com and verify your phone number (Settings → Phone verification). Kaggle asks for it before it gives GPU and Internet.

**Every time**
1. Open the session settings (**Settings** menu on a computer, **⋮ → Session options** on a phone): **Accelerator → GPU T4 x2** and **Internet → On**.
2. Run cell **1** (installs, about 3–5 minutes; nothing restarts).
3. Run cell **2**. The first time it downloads the voice model (a few GB). Then it shows **Open in Stickman Studio**: tap it on your phone and press **Connect** in the app. Or copy the link and paste it in Stickman Studio → **AI voice** → **Connect**.
4. Leave cell 2 running while you make voiceovers. When you are done, stop the session (power button / **Run → Stop session**) so your weekly GPU hours last.

Optional Hugging Face token (for the Hindi pack): add it under **Add-ons → Secrets** with the name `HF_TOKEN`; do not type it into the notebook.

Only clone your own voice, or a voice whose owner has given you permission. Audio made by Chatterbox carries Resemble AI's inaudible watermark.

In [ ]:
# 1 · Install (once per session, about 3-5 minutes)
import os, subprocess, sys, urllib.request
print('Stickman voice notebook for Kaggle v1')
W = '/kaggle/working'
try:
    urllib.request.urlopen('https://github.com', timeout=15)
except Exception:
    raise SystemExit('No internet. Session options -> Internet -> On (Kaggle asks you to verify your phone number first), then run this cell again.')
if os.path.exists(W + '/.ss_voice_installed_v1'):
    print('Already installed. Run cell 2.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'chatterbox-tts==0.1.7', 'fastapi', 'uvicorn', 'noisereduce', 'faster-whisper'], check=True)
    # Chatterbox pins PyTorch 2.6: use the matching torchvision ("torchvision::nms does not exist" otherwise)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision==0.21.0', '--no-deps'])
    subprocess.run(['wget', '-q', '-O', W + '/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    os.chmod(W + '/cloudflared', 0o755)
    open(W + '/.ss_voice_installed_v1', 'w').write('ok')
    print('Installed. Run cell 2.')

In [ ]:
# 2 · Start the voice server and get the link (leave this cell running)
BRANCH = "claude/stickman-video-voiceover-sync-ivss4s"
MODEL = "hindi"      # hindi, v3 or v2
HF_TOKEN = ""        # better: Add-ons -> Secrets -> HF_TOKEN
import os, re, json, signal, subprocess, sys, threading, time, urllib.request
print('Stickman voice notebook for Kaggle v1')
def show_link(url):
    '''Big button that opens the Stickman Studio app and offers to connect, plus the link to copy.'''
    from IPython.display import HTML, display
    import urllib.parse
    fallback = 'https://github.com/gopalkrishan680-hash/GK/releases/tag/android-latest'
    app = ('intent://connect?url=' + urllib.parse.quote(url, safe='') +
           '#Intent;scheme=stickmanstudio;package=com.gopalkrishan.stickmanstudio;S.browser_fallback_url=' +
           urllib.parse.quote(fallback, safe='') + ';end')
    display(HTML(f'''<div style="font:16px sans-serif;padding:14px;border:3px solid #f5c518;border-radius:10px">
      <b>Voice server is running.</b><br>
      <a href="{app}" target="_blank" style="display:inline-block;margin:12px 0;padding:14px 20px;background:#f5c518;color:#000;border-radius:10px;font-weight:bold;font-size:18px;text-decoration:none">&#128241; Open in Stickman Studio</a>
      <div>On a phone with the app: tap the button, then tap <b>Connect</b> in the app.<br>
      Otherwise copy this link (the app notices it when you switch back to it), or paste it in Stickman Studio &rarr; AI voice &rarr; Connect:</div>
      <input value="{url}" readonly onclick="this.select()" style="width:100%;font:18px monospace;margin-top:8px;padding:6px;box-sizing:border-box">
      <button onclick="navigator.clipboard.writeText('{url}').then(()=>this.textContent='Copied')" style="margin-top:8px;padding:8px 14px;font-size:16px">Copy link</button>
      <div style="margin-top:6px;color:#666">Keep this tab open while you make voiceovers.</div></div>'''))
    print(url)
W = '/kaggle/working'
if not os.path.exists(W + '/.ss_voice_installed_v1'):
    raise SystemExit('Run cell 1 first.')
if subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode != 0:
    raise SystemExit('No GPU. Session options -> Accelerator -> GPU T4 x2 (the session restarts), then run cells 1 and 2 again.')

# stop the server and tunnel of an earlier run of this cell
if os.path.exists(W + '/.ss_pids'):
    for pid in open(W + '/.ss_pids').read().split():
        try:
            os.kill(int(pid), signal.SIGKILL)
        except Exception:
            pass
    os.remove(W + '/.ss_pids')
    time.sleep(2)

# the server code of the newest commit (raw links of a branch name are cached for minutes)
try:
    ref = json.load(urllib.request.urlopen(f'https://api.github.com/repos/gopalkrishan680-hash/GK/commits/{BRANCH}', timeout=20))['sha']
except Exception:
    ref = BRANCH
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/gopalkrishan680-hash/GK/{ref}/stickman-voice/server.py?t={int(time.time())}', W + '/ss_server.py')
print('Server code commit', ref[:7])

if not HF_TOKEN.strip():
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
        if HF_TOKEN:
            print('Using HF_TOKEN from Kaggle Secrets')
    except Exception:
        HF_TOKEN = ''

# The server runs in its own process, so the freshly installed libraries load without a restart.
open(W + '/ss_run.py', 'w').write("import sys, time\nsys.path.insert(0, '/kaggle/working')\nimport ss_server\nprint('Server code version', ss_server.SERVER_VERSION, flush=True)\ne = ss_server.start(port=8000)\nprint('SS-READY', e.device, getattr(e, 'variant', '?'), ' | '.join(getattr(e, 'notes', None) or []), flush=True)\nwhile True:\n    time.sleep(3600)\n")
env = dict(os.environ, STICKMAN_MODEL=MODEL, PYTHONUNBUFFERED='1')
if HF_TOKEN.strip():
    env['HF_TOKEN'] = HF_TOKEN.strip()
LOG = W + '/ss_server.log'

def launch():
    return subprocess.Popen([sys.executable, '-u', W + '/ss_run.py'], stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, env=env)

print('Loading the voice model (first time: downloading a few GB)...')
srv, retried, shown, t0 = launch(), False, 0, time.time()
while True:
    time.sleep(3)
    text = open(LOG, errors='replace').read()
    lines = text.splitlines()
    for ln in lines[shown:]:
        if 'SS-READY' not in ln and re.search(r'Error|version|Loading|Note|Fix|Hindi|model', ln):
            print(ln[:300])
    shown = len(lines)
    if 'SS-READY' in text:
        break
    if srv.poll() is not None:
        if not retried and 'restarts now' in text:
            retried = True
            srv = launch()
            continue
        print('\n'.join(lines[-30:]))
        raise SystemExit('The voice server stopped (log above). Run this cell again; if it repeats, restart the session and run cells 1 and 2.')
    if time.time() - t0 > 1800:
        raise SystemExit('The model did not load in 30 minutes. Run this cell again.')
ready = [ln for ln in lines if 'SS-READY' in ln][-1].split(' ', 3)
print('Model ready on', ready[1], '· model:', ready[2])
if len(ready) > 3 and ready[3].strip():
    print('Note:', ready[3])

tunnel = subprocess.Popen([W + '/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
open(W + '/.ss_pids', 'w').write(f'{srv.pid} {tunnel.pid}')
url, t1 = None, time.time()
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
    if time.time() - t1 > 90:
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep draining the log
if not url:
    raise SystemExit('Could not open the public link. Run this cell again.')
show_link(url)

# Keep the cell running (keeps the Kaggle session active) and say if the server stops.
try:
    while True:
        time.sleep(60)
        if srv.poll() is not None:
            print('The voice server stopped. Run this cell again.')
            break
        if tunnel.poll() is not None:
            print('The public link closed. Run this cell again for a new link.')
            break
except KeyboardInterrupt:
    print('Stopped watching. The server keeps running until the session stops.')

### Troubleshooting
- **"No internet"**: Session options → Internet → On. Kaggle asks you to verify your phone number first (Settings → Phone verification).
- **"No GPU"**, or the GPU option is greyed out: verify your phone number and choose Accelerator **GPU T4 x2**. If your weekly GPU hours are used up, Kaggle shows it next to the accelerator; use Colab meanwhile.
- **The app cannot reach the server**: the link changes on every run; use the newest one, and keep cell 2 running.
- Everything else (robotic voice, Redo, sample cleaning): see the Troubleshooting section of the Colab notebook.